# Anomaly Detection in Financial Transactions

Let's apply AE to a **real-world industrial problem**: Fraud Detection.

**The Concept:**
1. We will train an Autoencoder **only** on "Normal" transactions.
2. The model becomes an expert at reconstructing normal spending habits.
3. When a "Fraudulent" (Anomalous) transaction appears, the model fails to reconstruct it.
4. This **Reconstruction Error** acts as an alarm.

We use labels to select only the 'Normal' data for training, but the model learns the features of that class in an unsupervised manner (reconstruction) without ever seeing an example of a 'Fraud' class during the training phase.

**Learning Objectives:**
- Working with tabular (CSV) data in PyTorch.
- Implementing the "Semi-supervised" anomaly detection workflow.
- Setting a statistical threshold for fraud detection.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Generating Synthetic Financial Data

We will create a dataset of 50 transactions
Most will be "Normal", but we will hide 5 "Fraudulent" cases.

In [ ]:
# Seed for reproducibility
np.random.seed(42)

# Names
names = ["Sunil Gurung", "Arun Patel", "Binita Thapa", "Priyanka Singh", "Prajwal Rai", 
         "Rajesh Sharma", "Shristi KC", "Vikram Reddy", "Ramesh Adhikari", "Ananya Gupta"]

# Generate 45 Normal Transactions, Normal = 0, Fraud = 1
normal_data = []
for i in range(45):
    name = np.random.choice(names)
    amount = np.random.randint(500, 5000)  # Standard daily spend
    hour = np.random.randint(8, 22)        # Daytime hours
    distance = np.random.uniform(0.5, 15.0) # Within same city
    category = np.random.choice([0, 1])    # Grocery or Utilities
    normal_data.append([name, amount, hour, distance, category, 0]) # 0 = Normal

# Generate 5 Anomalous Transactions (FRAUD)
fraud_data = [
    ["Sunil Gurung", 150000, 3, 2.0, 2, 1],    # Huge amount + 3 AM
    ["Binita Thapa", 2500, 14, 850.0, 2, 1],   # Unusual distance (850km)
    ["Vikram Reddy", 120000, 2, 80.0, 2, 1],   # Late night + Luxury
    ["Ananya Gupta", 50000, 23, 1.0, 2, 1],    # High amount + Midnight
    ["Arun Patel", 3000, 15, 1200.0, 0, 1]     # Extreme distance
]

# Combine into DataFrame
columns = ["User_Name", "Amount", "Hour", "Dist_Home_KM", "Category", "Is_Fraud"]
df = pd.DataFrame(normal_data + fraud_data, columns=columns)

# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True) # frac=1 means 100% of the data
# df = df.shuffle().reset_index(drop=True)

# Save to check
df.head(10)

## 3. Data Preprocessing

For Autoencoders to work on tabular data, we **must** scale the features.

Imagine having `Amount` (5000) and `Category` (0.1). 

Without scaling, the model will think the Amount is 50,000 times more important than the Category!

In [ ]:
# Separate Features and Ground Truth
# We drop 'User_Name' as it's just meta-data
features = df.drop(columns=["User_Name", "Is_Fraud"])
labels = df["Is_Fraud"]

# Scale the features (Mean=0, Std=1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# convert to PyTorch tensors
X_tensor = torch.FloatTensor(features_scaled)

# Preparation for "Semi-Supervised" Learning:
train_indices = torch.tensor(
    df[df["Is_Fraud"] == 0].index.values,
    dtype=torch.long
)
X_train = X_tensor[train_indices]

# The test set now contains BOTH Normal and Fraud
X_test = X_tensor

print(f"Training set size (Normal only): {len(X_train)}")
print(f"Test set size (Mixed): {len(X_test)}")

## 4. The Tabular Autoencoder

A simple fully-connected MLP.
- Input (4 features)
- Hidden (8)
- Bottleneck (2)
- Hidden (8)
- Output (4)

In [ ]:
class TabularAE(nn.Module):
    def __init__(self, input_dim=4):
        super(TabularAE, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 2) # Latent size 2
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim)
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model = TabularAE().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
model

In [ ]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

## 5. Training the "Expert"

We train the model **ONLY** on the normal data. It will learn the pattern: 
*"Normal transactions are small, during day, and close to home."*

In [ ]:
num_epochs = 100
train_loader = DataLoader(X_train, batch_size=4, shuffle=True)

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for data in train_loader:
        data = data.to(device)
        
        # Forward
        output = model(data)
        loss = criterion(output, data)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

## 6. Detecting Anomalies

Now we pass the **Full Test Set** (Normal + Fraud) through the model. 
We calculate the **Mean Squared Error (MSE)** for every single row.

In [ ]:
model.eval()
with torch.no_grad():
    reconstructed = model(X_test.to(device))

    # Calculate MSE for each individual row
    mse_per_row = torch.mean((X_test.to(device) - reconstructed)**2, dim=1)

# Add to the original DataFrame for analysis
df['Reconstruction_Error'] = mse_per_row.tolist()

# Let's see the error for Fraud vs Normal
print("Average Error for Normal Transactions:", df[df['Is_Fraud']==0]['Reconstruction_Error'].mean())
print("Average Error for Fraud Transactions:", df[df['Is_Fraud']==1]['Reconstruction_Error'].mean())

### Setting the Threshold

We set a threshold. If Error > Threshold, we sound the alarm.

In [ ]:
mean_err = df[df['Is_Fraud']==0]['Reconstruction_Error'].mean()
std_err = df[df['Is_Fraud']==0]['Reconstruction_Error'].std()

# Threshold = Mean + 3 Std Deviations
threshold = mean_err + 3 * std_err

print(f"Statistical Threshold: {threshold:.4f}")

# Visualize
plt.figure(figsize=(10, 6))
plt.hist(df[df['Is_Fraud']==0]['Reconstruction_Error'], bins=20, alpha=0.5, label='Normal', color='blue')
plt.hist(df[df['Is_Fraud']==1]['Reconstruction_Error'], bins=20, alpha=0.7, label='Fraud', color='red')
plt.axvline(threshold, color='green', linestyle='--', label='Alarm Threshold')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.title('Fraud vs Normal Error Distribution')
plt.legend()
plt.show()

## 7. Identifying the "Fraudsters"

Let's see who the model flagged as high risk.

In [ ]:
# Flag items above threshold
df['Predicted_Fraud'] = df['Reconstruction_Error'] > threshold

# Show the results
flagged = df[df['Predicted_Fraud'] == True]
print(f"Total transactions flagged: {len(flagged)}")
flagged[['User_Name', 'Amount', 'Hour', 'Dist_Home_KM', 'Is_Fraud', 'Reconstruction_Error']]

## 6.5 Visualizing the Latent Space

Since our bottleneck is size 2, we can directly plot the latent coordinates ($z_1, z_2$) to see how the model separates normal transactions from anomalies.

In [ ]:
model.eval()
with torch.no_grad():
    # Get latent representations (encoder output)
    latent_reps = model.encoder(X_test.to(device))

In [ ]:
# latent space
latent_reps.shape

In [ ]:
# Plot Normal points

plt.figure(figsize=(10, 7))
plt.scatter(latent_reps[df['Is_Fraud']==0, 0], latent_reps[df['Is_Fraud']==0, 1], 
            alpha=0.6, label='Normal', color='blue', edgecolor='k')

# Plot Fraud points

plt.scatter(latent_reps[df['Is_Fraud']==1, 0], latent_reps[df['Is_Fraud']==1, 1], 
            alpha=0.9, label='Fraud', color='red', marker='X', s=100, edgecolor='black')

plt.title('Latent Space Visualization (Bottleneck)', fontsize=14, fontweight='bold')
plt.xlabel('Latent Feature 1', fontsize=12)
plt.ylabel('Latent Feature 2', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7.5 Deep Dive: Why was it flagged?

Llook at one flagged transaction and see which specific feature (Amount, Hour, etc.) contributed most to the high reconstruction error.

In [ ]:
# Pick the highest error transaction
top_fraud_idx = df['Reconstruction_Error'].idxmax()
original_row = X_test[top_fraud_idx].to(device)

model.eval()
with torch.no_grad():
    recon_row = model(original_row.unsqueeze(0)).squeeze()

# Calculate squared error per feature
feature_errors = (original_row - recon_row)**2
feature_names = features.columns

plt.figure(figsize=(8, 5))
plt.bar(feature_names, feature_errors.cpu().numpy(), color='orange', edgecolor='black')
plt.title(f"Error Breakdown for User: {df.loc[top_fraud_idx, 'User_Name']}", fontsize=13)
plt.ylabel("Reconstruction Error (Squared)")
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"User Details: {df.loc[top_fraud_idx, ['Amount', 'Hour', 'Dist_Home_KM']].to_dict()}")
# Insight: The bar with the highest value is the 'reason' the model felt the transaction was weird!

## 8. Student Exercises & Challenges

### Task 1: The "Sensitivity" Trade-off
Currently, we use `Threshold = Mean + 3*Std`.
- **Action**: Change the multiplier from `3` to `1.5` and then to `5`.
- **Question**: How many "Normal" people did you accidentally accuse of fraud (False Positives) when the threshold was `1.5`? 
- **Learning**: This is the balance between Security (Catching all fraud) and Customer Experience (Not blocking real users).

### Task 2: Detecting the "Stealthy" Fraud
Add a new row to the data that is "slightly" abnormal but not extreme (e.g., Amount=15,000 at 5 PM).
- **Action**: Does the model catch it?
- **Task**: Try increasing the **Bottleneck size** from 2 to 3. Does a larger bottleneck make the model better or worse at catching "subtle" fraud? Why?

### Task 3: Feature Impact
In Section 7.5, we calculated error per feature.
- **Action**: Create a loop to identify the "Primary Reason for Alarm" for all 5 fraud cases. 
- **Output**: Print a summary like: *"User Rajesh was flagged primarily because of 'Distance', whereas Binita was flagged due to 'Amount'."*

### Task 4: VAE vs AE for Anomaly
- **Concept**: A VAE (from Workshop 2) provides a probability distribution.
- **Task**: If you used a VAE here, you could use the **Log-Likelihood** as an anomaly score instead of MSE. 
- **Challenge**: Try implementing a simple `VAE` for this same tabular data. Does the "latent space" look compact than the standard AE?